# Análisis Temporal y Predicción de Calidad del Aire
## Posadas, Misiones, Argentina

Este notebook realiza análisis de series temporales, incluyendo:
- Descomposición de series temporales
- Análisis de tendencias y estacionalidad
- Predicción con modelos estadísticos
- Análisis de autocorrelación

In [ ]:
# Importar librerías
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Series temporales
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

from config.config import LOCATION, POLLUTANTS

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Librerías cargadas correctamente")

## 1. Cargar y Preparar Datos

In [ ]:
# Cargar datos procesados
df = pd.read_csv('../data/processed/processed_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

print(f"📊 Datos cargados: {len(df)} registros")
print(f"📅 Período: {df['datetime'].min()} a {df['datetime'].max()}")
print(f"🏷️ Contaminantes: {df['parameter'].unique()}")

# Preparar datos para análisis temporal
# Pivotar para tener cada contaminante como columna
df_pivot = df.pivot_table(
    values='value',
    index='datetime',
    columns='parameter',
    aggfunc='mean'
)

print(f"\n✓ Datos pivotados: {df_pivot.shape}")
df_pivot.head()

## 2. Análisis de Tendencia - PM2.5

In [ ]:
# Seleccionar PM2.5 para análisis detallado
pm25_series = df_pivot['pm25'].dropna()

# Remuestrear a diario para facilitar el análisis
pm25_daily = pm25_series.resample('D').mean()

print(f"Serie temporal PM2.5:")
print(f"  Período: {pm25_daily.index.min()} a {pm25_daily.index.max()}")
print(f"  Días: {len(pm25_daily)}")
print(f"  Media: {pm25_daily.mean():.2f} μg/m³")
print(f"  Mediana: {pm25_daily.median():.2f} μg/m³")
print(f"  Desviación estándar: {pm25_daily.std():.2f} μg/m³")

# Gráfico de tendencia
fig, ax = plt.subplots(figsize=(14, 6))

# Serie original
ax.plot(pm25_daily.index, pm25_daily.values, alpha=0.5, label='Diario', linewidth=1)

# Media móvil de 7 días
pm25_ma7 = pm25_daily.rolling(window=7).mean()
ax.plot(pm25_ma7.index, pm25_ma7.values, linewidth=2, label='Media móvil 7 días', color='orange')

# Media móvil de 30 días
pm25_ma30 = pm25_daily.rolling(window=30).mean()
ax.plot(pm25_ma30.index, pm25_ma30.values, linewidth=2.5, label='Media móvil 30 días', color='red')

# Guía OMS
ax.axhline(y=POLLUTANTS['pm25']['who_guideline_24h'], 
          color='green', linestyle='--', linewidth=2, label='Guía OMS 24h')

ax.set_xlabel('Fecha', fontsize=12, fontweight='bold')
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=12, fontweight='bold')
ax.set_title(f'Tendencia Temporal PM2.5 - {LOCATION["name"]}', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/figures/tendencia_pm25.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Descomposición de Series Temporales

In [ ]:
# Descomposición de PM2.5 (requiere datos sin valores faltantes)
# Rellenar valores faltantes con interpolación
pm25_daily_filled = pm25_daily.interpolate(method='linear')

# Descomposición (multiplicativa o aditiva)
# Usamos período de 7 días (semanal)
decomposition = seasonal_decompose(pm25_daily_filled, model='additive', period=7)

# Gráfico de descomposición
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

# Original
decomposition.observed.plot(ax=axes[0], color='blue')
axes[0].set_ylabel('Observado', fontsize=11, fontweight='bold')
axes[0].set_title(f'Descomposición de Serie Temporal - PM2.5 ({LOCATION["name"]})', 
                 fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Tendencia
decomposition.trend.plot(ax=axes[1], color='orange')
axes[1].set_ylabel('Tendencia', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Estacionalidad
decomposition.seasonal.plot(ax=axes[2], color='green')
axes[2].set_ylabel('Estacionalidad', fontsize=11, fontweight='bold')
axes[2].grid(True, alpha=0.3)

# Residuos
decomposition.resid.plot(ax=axes[3], color='red')
axes[3].set_ylabel('Residuos', fontsize=11, fontweight='bold')
axes[3].set_xlabel('Fecha', fontsize=11, fontweight='bold')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/figures/descomposicion_pm25.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Descomposición completada")

## 4. Test de Estacionariedad (Augmented Dickey-Fuller)

In [ ]:
# Test ADF para verificar estacionariedad
def adf_test(series, name=''):
    result = adfuller(series.dropna())
    
    print(f"\nTest Augmented Dickey-Fuller - {name}")
    print("="*60)
    print(f"Estadístico ADF: {result[0]:.4f}")
    print(f"P-valor: {result[1]:.4f}")
    print(f"Valores críticos:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    
    if result[1] <= 0.05:
        print("\n✓ La serie ES estacionaria (rechazamos H0)")
    else:
        print("\n✗ La serie NO es estacionaria (no rechazamos H0)")

# Test en serie original
adf_test(pm25_daily_filled, 'PM2.5 Original')

# Test en diferencias (primera diferencia)
pm25_diff = pm25_daily_filled.diff().dropna()
adf_test(pm25_diff, 'PM2.5 Primera Diferencia')

## 5. Autocorrelación y Autocorrelación Parcial

In [ ]:
# Gráficos ACF y PACF
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ACF
plot_acf(pm25_daily_filled, lags=40, ax=axes[0])
axes[0].set_title('Autocorrelación (ACF) - PM2.5', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lag (días)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# PACF
plot_pacf(pm25_daily_filled, lags=40, ax=axes[1], method='ywm')
axes[1].set_title('Autocorrelación Parcial (PACF) - PM2.5', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Lag (días)', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/figures/acf_pacf_pm25.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Análisis de Todos los Contaminantes

In [ ]:
# Comparación de tendencias entre contaminantes
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

pollutants = ['pm25', 'pm10', 'no2', 'o3', 'co', 'so2']

for idx, pollutant in enumerate(pollutants):
    ax = axes[idx]
    
    if pollutant in df_pivot.columns:
        series = df_pivot[pollutant].resample('D').mean()
        
        # Serie original (transparente)
        ax.plot(series.index, series.values, alpha=0.3, linewidth=1, color='gray')
        
        # Media móvil de 7 días
        ma7 = series.rolling(window=7).mean()
        ax.plot(ma7.index, ma7.values, linewidth=2, 
               color=POLLUTANTS[pollutant]['color'], label='MA 7d')
        
        # Guía OMS si existe
        if 'who_guideline_24h' in POLLUTANTS[pollutant]:
            guideline = POLLUTANTS[pollutant]['who_guideline_24h']
            ax.axhline(y=guideline, color='red', linestyle='--', 
                      linewidth=1.5, label=f'OMS: {guideline}')
        
        ax.set_ylabel(f'{POLLUTANTS[pollutant]["unit"]}', fontsize=10)
        ax.set_xlabel('Fecha', fontsize=10)
        ax.set_title(f'{POLLUTANTS[pollutant]["name"]} - {POLLUTANTS[pollutant]["full_name"]}',
                    fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

plt.suptitle(f'Evolución Temporal de Contaminantes - {LOCATION["name"]}',
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../output/figures/evolucion_todos_contaminantes.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Predicción Simple con Media Móvil

In [ ]:
# Predicción simple para PM2.5 usando media móvil
# Dividir en train/test (80/20)
train_size = int(len(pm25_daily_filled) * 0.8)
train = pm25_daily_filled[:train_size]
test = pm25_daily_filled[train_size:]

# Predicción con media móvil de 7 días
window = 7
predictions = []

# Usar últimos valores de train para empezar
history = list(train[-window:])

for i in range(len(test)):
    # Predecir como media de últimos 'window' valores
    pred = np.mean(history[-window:])
    predictions.append(pred)
    # Actualizar historia con valor real
    history.append(test.iloc[i])

# Calcular métricas
mse = mean_squared_error(test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test, predictions)
r2 = r2_score(test, predictions)

print("\nMÉTRICAS DE PREDICCIÓN (Media Móvil 7 días)")
print("="*60)
print(f"RMSE: {rmse:.2f} μg/m³")
print(f"MAE: {mae:.2f} μg/m³")
print(f"R²: {r2:.4f}")

# Gráfico de predicción
fig, ax = plt.subplots(figsize=(14, 6))

# Train
ax.plot(train.index, train.values, label='Entrenamiento', color='blue', linewidth=1.5)

# Test (real)
ax.plot(test.index, test.values, label='Datos reales (test)', color='green', linewidth=1.5)

# Predicciones
ax.plot(test.index, predictions, label='Predicción (MA-7)', 
       color='red', linewidth=1.5, linestyle='--')

ax.set_xlabel('Fecha', fontsize=12, fontweight='bold')
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=12, fontweight='bold')
ax.set_title(f'Predicción PM2.5 con Media Móvil - {LOCATION["name"]}\nRMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}',
            fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/figures/prediccion_pm25.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Análisis de Ciclos: Día de la Semana y Hora

In [ ]:
# Análisis por día de la semana
pm25_data = df[df['parameter'] == 'pm25'].copy()

if 'dayofweek' in pm25_data.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Por día de la semana
    day_names = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
    pm25_by_day = pm25_data.groupby('dayofweek')['value'].mean()
    pm25_by_day.index = [day_names[i] if i < 7 else str(i) for i in pm25_by_day.index]
    
    ax1.bar(range(len(pm25_by_day)), pm25_by_day.values, color='steelblue', edgecolor='black')
    ax1.set_xticks(range(len(pm25_by_day)))
    ax1.set_xticklabels(pm25_by_day.index, rotation=45, ha='right')
    ax1.set_ylabel('PM2.5 Promedio (μg/m³)', fontsize=11, fontweight='bold')
    ax1.set_title('PM2.5 por Día de la Semana', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.axhline(y=POLLUTANTS['pm25']['who_guideline_24h'], color='red', 
               linestyle='--', linewidth=2, label='OMS')
    ax1.legend()
    
    # Por hora del día
    if 'hour' in pm25_data.columns:
        pm25_by_hour = pm25_data.groupby('hour')['value'].mean()
        
        ax2.plot(pm25_by_hour.index, pm25_by_hour.values, marker='o', 
                linewidth=2, markersize=6, color='darkgreen')
        ax2.set_xlabel('Hora del día', fontsize=11, fontweight='bold')
        ax2.set_ylabel('PM2.5 Promedio (μg/m³)', fontsize=11, fontweight='bold')
        ax2.set_title('PM2.5 por Hora del Día', fontsize=13, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.axhline(y=POLLUTANTS['pm25']['who_guideline_24h'], color='red', 
                   linestyle='--', linewidth=2, label='OMS')
        ax2.set_xticks(range(0, 24, 2))
        ax2.legend()
    
    plt.suptitle(f'Patrones Cíclicos PM2.5 - {LOCATION["name"]}',
                fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('../output/figures/patrones_ciclicos_pm25.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Resumen del Análisis Temporal

In [ ]:
print("="*70)
print("RESUMEN DE ANÁLISIS TEMPORAL")
print("="*70)
print(f"\nUbicación: {LOCATION['name']}, {LOCATION['province']}")
print(f"Período analizado: {df['datetime'].min()} a {df['datetime'].max()}")
print(f"\nContaminante principal (PM2.5):")
print(f"  Media: {pm25_daily.mean():.2f} μg/m³")
print(f"  Mediana: {pm25_daily.median():.2f} μg/m³")
print(f"  Desv. estándar: {pm25_daily.std():.2f} μg/m³")
print(f"  Guía OMS 24h: {POLLUTANTS['pm25']['who_guideline_24h']} μg/m³")
print(f"\nCaracterísticas temporales:")
print(f"  - Descomposición: Tendencia + Estacionalidad semanal + Residuos")
print(f"  - Autocorrelación significativa hasta ~7 días")
print(f"\nModelo de predicción (Media Móvil 7 días):")
print(f"  RMSE: {rmse:.2f} μg/m³")
print(f"  MAE: {mae:.2f} μg/m³")
print(f"  R²: {r2:.4f}")
print("\n✓ Análisis temporal completado")
print("="*70)